# Open-loop gait baseline

Hand-designed foot trajectory + leg inverse kinematics executed in CoppeliaSim. 
Used as a reference point for the RL policies (cumulated reward computed with the same reward function).

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
from MovementTrajectoryGenerator import TrajectoryGenerator

In [ ]:
def average_cycles(signal, cycle_len):
    signal = np.array(signal)

    num_cycles = len(signal) // cycle_len
    trimmed = signal[:num_cycles * cycle_len]

    cycles = trimmed.reshape(num_cycles, cycle_len)
    mean_cycle = cycles.mean(axis=0)

    return mean_cycle

In [ ]:
tc = TrajectoryGenerator(step_frequency=0.85, step_length=0.08, step_height=0.06, step_offset=0.03)
tc.reset()
tc.plot_trajectory(sample_period_ms=50)

In [ ]:
t = 0.0
dt = tc.number_of_sim_steps_between_control_steps*0.05
print(f'dt {dt}')
cumulated_reward = 0.0
left_knee_actions_list = []
left_hip_actions_list = []

move_types = ["knees", "hips","knees","None"]
action = tc.generate_action(t)

for step in range(300):
    for move_type in move_types:
        generated_actions = tc.generate_action(t)
        if move_type == "knees":
            action[1] = generated_actions[1]
            action[3] = generated_actions[3]
            action[5] = generated_actions[5]
            action[7] = generated_actions[7]
        elif move_type == "hips":
            action[0] = generated_actions[0]
            action[2] = generated_actions[2]
            action[4] = generated_actions[4]
            action[6] = generated_actions[6]
        
        RobotBasePosition = tc.sim.getObjectPosition(tc.RobotBaseHandle,-1)
        RobotBaseOrientation = tc.sim.getObjectOrientation(tc.RobotBaseHandle,-1)
        velocity = tc.sim.getObjectVelocity(tc.RobotBaseHandle)[0][0]

        if step%2 == 0:
            obs_actor = [
            action[0], action[1], action[2], action[3],
            action[4], action[5], action[6], action[7],
            tc.from_gyroscope_to_observation_space(RobotBaseOrientation[0]), tc.from_gyroscope_to_observation_space(RobotBaseOrientation[1])
            ]
            tc.observation_history_metric.append(obs_actor)

            reward = tc.calculate_reward(RobotBasePosition, RobotBaseOrientation, velocity, None)
            cumulated_reward += reward

        tc.execute_action(action.tolist(), movement_type=move_type)
        tc.make_simulation_steps(tc.number_of_sim_steps_between_control_steps)
        
        tc.get_joint_torques()
        
        t += dt
        left_hip_actions_list.append(action[2])
        left_knee_actions_list.append(action[3])

final_pos = tc.sim.getObjectPosition(tc.RobotBaseHandle,-1)[0]
print("Final position:", final_pos)
print("Velocity estimate:", final_pos / t)
tc.reset()
print("Cumulated reward:", cumulated_reward)

In [ ]:
tc.print_min_max_actions()

In [ ]:
cycle_len = 40

In [ ]:
tc.plot_torques()

In [ ]:
left_knee_actions_avg = average_cycles(left_knee_actions_list, cycle_len=cycle_len)
left_hip_actions_avg = average_cycles(left_hip_actions_list, cycle_len=cycle_len)

In [ ]:
#make 2 plots one above another for left hip and left knee angles over time
plt.figure(figsize=(10, 8))
plt.subplot(2, 1, 1)
plt.step(np.arange(len(left_hip_actions_avg))*dt, np.rad2deg(left_hip_actions_avg), label='Left Hip Angle')
plt.grid()
plt.title('Left Hip Angle over Time')
plt.xlabel('Time (s)')
plt.ylabel('Angle (degrees)')
plt.subplot(2, 1, 2)
plt.step(np.arange(len(left_knee_actions_avg))*dt, np.rad2deg(left_knee_actions_avg), label='Left Knee Angle', color='orange')
plt.grid()
plt.title('Left Knee Angle over Time')
plt.xlabel('Time (s)')
plt.ylabel('Angle (degrees)')   

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(np.rad2deg(tc.hip_action_list), bins=20, color='blue', alpha=0.7)
plt.title('Hip Angles Histogram')
plt.subplot(1, 2, 2)
plt.hist(np.rad2deg(tc.knee_action_list), bins=20, color='orange', alpha=0.7)
plt.title('Knee Angles Histogram')
plt.show()

In [ ]:
tc.stop_simulation()